<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/notebooks/data_setup_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')   # not secret, fine to type visibly
os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')         # secret, hidden as you type

!kaggle datasets download -d mlg-ulb/creditcardfraud -p /content/data/ulb --unzip
!kaggle datasets download -d kartik2112/fraud-detection -p /content/data/sparkov --unzip

Kaggle username: Subbareddy_Subbz
Kaggle API key: ··········
Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
100% 66.0M/66.0M [00:03<00:00, 18.3MB/s]

Dataset URL: https://www.kaggle.com/datasets/kartik2112/fraud-detection
License(s): CC0-1.0
100% 202M/202M [00:10<00:00, 20.9MB/s]



In [3]:
from getpass import getpass

github_token = getpass('Paste your GitHub token: ')
repo_url = f'https://{github_token}@github.com/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-.git'

!git clone {repo_url}

Paste your GitHub token: ··········
Cloning into 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 31 (delta 10), reused 11 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 150.79 KiB | 1.51 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [4]:
%cd Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-
!ls

/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-
data  notebooks  README.md  requirements.txt  results  src


In [5]:
!mv /content/data/ulb /content/data/sparkov data/
!ls data

sparkov  ulb


In [6]:
!ls src
!cat src/split.py

split.py
"""
Train/test split for the fraud detection pipeline (Stage 4 of our methodology).

Run this once. Every later stage - EDA, feature engineering, baseline
training, model comparison, ablation - reads from data/processed/ instead
of splitting the data again. Keeping the split in one place is what
actually makes the leak-free guarantee hold across the rest of the
project; re-deriving it in different notebooks is how that guarantee
quietly breaks.

ULB doesn't come with a train/test split, so we cut it ourselves along
time (see chronological_split() below). Sparkov is different - Kaggle
already ships fraudTrain.csv and fraudTest.csv, and we didn't take that
on faith before using it. We checked: no overlap between the two files'
transaction times (train's last transaction is 2020-06-21 12:13:37,
test's first is 2020-06-21 12:14:25), zero transaction IDs shared between
them, zero duplicate rows in either file. Given that, we use Kaggle's
split as-is instead of re-splitting fraudTra

In [7]:
%%writefile src/split.py
"""
Train/test split for the fraud detection pipeline (Stage 4 of our methodology).

Run this once. Every later stage - EDA, feature engineering, baseline
training, model comparison, ablation - reads from data/processed/ instead
of splitting the data again. Keeping the split in one place is what
actually makes the leak-free guarantee hold across the rest of the
project; re-deriving it in different notebooks is how that guarantee
quietly breaks.

ULB doesn't come with a train/test split, so we cut it ourselves along
time (see chronological_split() below). Sparkov is different - Kaggle
already ships fraudTrain.csv and fraudTest.csv, and we didn't take that
on faith before using it. We checked: no overlap between the two files'
transaction times (train's last transaction is 2020-06-21 12:13:37,
test's first is 2020-06-21 12:14:25), zero transaction IDs shared between
them, zero duplicate rows in either file. Given that, we use Kaggle's
split as-is instead of re-splitting fraudTrain.csv ourselves, which
would've thrown away all 555,719 rows of fraudTest.csv for nothing.
"""

import pandas as pd
from pathlib import Path

# Where the raw Kaggle data lives, and where the split files get saved
RAW_DIR = Path("data")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def chronological_split(df, time_column, train_fraction=0.8):
    """
    Splits a dataframe into train/test by TIME, not randomly - everything
    before the cutoff goes to train, everything after goes to test.
    Used for ULB, which has no ready-made split.
    """
    # Step 1: put every row in true time order
    df = df.sort_values(time_column).reset_index(drop=True)

    # Step 2: find the row that marks 80% of the way through
    cutoff_row = int(len(df) * train_fraction)

    # Step 3: everything before that row is train, everything after is test
    train = df.iloc[:cutoff_row].copy()
    test = df.iloc[cutoff_row:].copy()

    print(f"Train: {len(train):,} rows, from {train[time_column].min()} to {train[time_column].max()}")
    print(f"Test:  {len(test):,} rows, from {test[time_column].min()} to {test[time_column].max()}")

    # Safety check: train must fully end before test begins
    assert train[time_column].max() <= test[time_column].min(), "Something's wrong - the split is not chronological!"

    return train, test


def split_sparkov():
    """
    Loads Sparkov's train/test files exactly as Kaggle provides them.
    We're not splitting anything here - we already checked by hand that
    Kaggle's files are clean and chronological, so we just load and save them.
    """
    train = pd.read_csv(RAW_DIR / "sparkov" / "fraudTrain.csv")
    test = pd.read_csv(RAW_DIR / "sparkov" / "fraudTest.csv")

    train["trans_date_trans_time"] = pd.to_datetime(train["trans_date_trans_time"])
    test["trans_date_trans_time"] = pd.to_datetime(test["trans_date_trans_time"])

    # These checks should always pass - we already verified this by hand.
    # If they ever fail, something changed and needs a look.
    assert train["trans_date_trans_time"].max() < test["trans_date_trans_time"].min(), \
        "Train and test transaction times overlap - stop and check!"

    shared_ids = set(train["trans_num"]) & set(test["trans_num"])
    assert len(shared_ids) == 0, "Found the same transaction in both train and test - stop and check!"

    print(f"Train: {len(train):,} rows, from {train['trans_date_trans_time'].min()} to {train['trans_date_trans_time'].max()}")
    print(f"Test:  {len(test):,} rows, from {test['trans_date_trans_time'].min()} to {test['trans_date_trans_time'].max()}")

    train.to_csv(PROCESSED_DIR / "sparkov_train.csv", index=False)
    test.to_csv(PROCESSED_DIR / "sparkov_test.csv", index=False)

    return train, test


def split_ulb():
    """Loads ULB, removes duplicate rows, splits it by time, and saves the result."""
    ulb = pd.read_csv(RAW_DIR / "ulb" / "creditcard.csv")
    ulb = ulb.drop_duplicates()  # duplicates found during EDA

    # ULB's "Time" column is just seconds since the first transaction,
    # so it's already something we can sort by directly
    train, test = chronological_split(ulb, time_column="Time", train_fraction=0.8)

    train.to_csv(PROCESSED_DIR / "ulb_train.csv", index=False)
    test.to_csv(PROCESSED_DIR / "ulb_test.csv", index=False)

    return train, test


# Lets this file be run directly from a terminal with: python src/split.py
if __name__ == "__main__":
    print("Splitting Sparkov...")
    split_sparkov()

    print("\nSplitting ULB...")
    split_ulb()

Overwriting src/split.py


In [8]:
import sys
sys.path.append('/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-')

from src.split import split_sparkov, split_ulb

print("Splitting Sparkov...")
sparkov_train, sparkov_test = split_sparkov()

print("\nSplitting ULB...")
ulb_train, ulb_test = split_ulb()

Splitting Sparkov...
Train: 1,296,675 rows, from 2019-01-01 00:00:18 to 2020-06-21 12:13:37
Test:  555,719 rows, from 2020-06-21 12:14:25 to 2020-12-31 23:59:34

Splitting ULB...
Train: 226,980 rows, from 0.0 to 145233.0
Test:  56,746 rows, from 145234.0 to 172792.0


In [9]:
%%writefile src/utils.py
"""
Shared helper functions for the fraud detection pipeline.

split.py, the EDA notebook, and the feature-engineering script (coming
next) all need the same handful of calculations - distance between two
GPS points, how long since a card's last transaction, how fast someone
would've had to travel between transactions, and so on. Instead of
writing these separately in every file and risking them drifting out of
sync, we define them once here and import them wherever they're needed.
"""

import numpy as np
import pandas as pd


def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Straight-line distance in km between two GPS points, accounting for
    the Earth being a sphere rather than a flat plane. Used to measure
    how far a transaction happened from a cardholder's home, and how far
    apart two consecutive transactions were.
    """
    R = 6371  # Earth's radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c


def add_prev_txn_features(df, id_column, time_column, lat_column, lon_column):
    """
    Adds three columns to a transactions dataframe, each comparing a row
    to that same cardholder's previous transaction:

      - time_gap_hours        : hours since the cardholder's last transaction
      - dist_from_prev_txn_km : how far this transaction is from the last one
      - implied_velocity_kmh  : distance divided by time - "how fast would
                                 this person have had to travel?"

    All three only ever look backward in time for a given cardholder, so
    they're safe to compute on the full dataset before splitting - each
    row's value only depends on that same cardholder's past, never on
    anyone else's future.
    """
    df = df.sort_values([id_column, time_column]).copy()

    prev_lat = df.groupby(id_column)[lat_column].shift(1)
    prev_lon = df.groupby(id_column)[lon_column].shift(1)
    prev_time = df.groupby(id_column)[time_column].shift(1)

    df["dist_from_prev_txn_km"] = haversine_distance(df[lat_column], df[lon_column], prev_lat, prev_lon)
    df["time_gap_hours"] = (df[time_column] - prev_time).dt.total_seconds() / 3600

    # add a tiny amount of time (1 second) so we never divide by zero
    # for two transactions that happened at the exact same instant
    epsilon = 1 / 3600
    df["implied_velocity_kmh"] = df["dist_from_prev_txn_km"] / (df["time_gap_hours"] + epsilon)

    return df


def add_rolling_txn_count(df, id_column, time_column, window="24h", amount_column="amt"):
    """
    Adds a column counting how many transactions this cardholder made in
    the trailing time window (default: past 24 hours), up to and
    including the current transaction. Backward-looking, same as above -
    safe to compute before splitting.
    """
    df = df.sort_values([id_column, time_column]).copy()
    df_indexed = df.set_index(time_column)

    column_name = f"txn_count_{window}"
    df[column_name] = (
        df_indexed.groupby(id_column)[amount_column]
        .rolling(window)
        .count()
        .reset_index(level=0, drop=True)
        .values
    )
    return df


def fit_category_risk(train_df, category_column, label_column):
    """
    Calculates the fraud rate for each merchant category, using TRAIN
    DATA ONLY. This is different from the functions above: it's built
    from the fraud label itself, so if we calculated it on the full
    dataset (train + test), we'd be leaking test-set fraud information
    into a feature the model sees during training. Fit here, then apply
    the same frozen numbers to test with apply_category_risk() below.
    """
    risk_by_category = train_df.groupby(category_column)[label_column].mean().to_dict()
    fallback_rate = train_df[label_column].mean()  # used for any category test has that train never saw
    return {"risk_by_category": risk_by_category, "fallback_rate": fallback_rate}


def apply_category_risk(df, category_column, fitted):
    """Applies a risk lookup produced by fit_category_risk() to any dataframe."""
    return df[category_column].map(fitted["risk_by_category"]).fillna(fitted["fallback_rate"])


def id_overlap_report(train_df, test_df, id_column):
    """
    Reports how many cardholders appear in both train and test. This is
    just informational, not a red flag - for our chronological split,
    the same cardholder legitimately shows up in both (their earlier
    transactions in train, later ones in test). Recording this here
    means it's a documented, expected fact instead of a surprise later.
    """
    train_ids = set(train_df[id_column].unique())
    test_ids = set(test_df[id_column].unique())
    overlap = train_ids & test_ids

    return {
        "cardholders_in_train": len(train_ids),
        "cardholders_in_test": len(test_ids),
        "cardholders_in_both": len(overlap),
    }

Writing src/utils.py


In [11]:
!git add src/split.py src/utils.py
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   src/utils.py



In [13]:
!git config --global user.email "subbareddy.pullalacheruvu@students.mq.edu.au"
!git config --global user.name "subbareddypullalacheruvu"

In [14]:
!git commit -m "Add shared helper functions (haversine distance, sequence features, category-risk fit/apply)"
!git push

[main 8fc6dbc] Add shared helper functions (haversine distance, sequence features, category-risk fit/apply)
 1 file changed, 120 insertions(+)
 create mode 100644 src/utils.py
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 2.41 KiB | 2.41 MiB/s, done.
Total 4 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-.git
   ca2a7a8..8fc6dbc  main -> main
